# CSV: rule-only structural families

Experimental notebook interface to `extension.clustering.mispricing.structural`. The existing implementation remains authoritative. Run cells from top to bottom in a fresh kernel. Outputs are isolated under `output/notebook_extension/`; rerunning replaces that notebook’s outputs.

This is a staged orchestration of existing functions, not an independent reimplementation. Open a function’s source to change its algorithm; substitute a compatible function call below to test a variation. Preserve train-only fitting, splits, matched costs, support thresholds and dependence-aware uncertainty. See the repository methodology before interpreting results.

## 1. Locate the project and import reusable functions
The Python environment must contain the project dependencies plus a Jupyter kernel.

In [ ]:
import os
import sys
from pathlib import Path

# Works when Jupyter starts in this folder or anywhere inside the repository.
ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "extension/pipeline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the project repository.")
os.chdir(ROOT)
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
os.environ.setdefault("MPLBACKEND", "Agg")
os.environ.setdefault("MPLCONFIGDIR", "/tmp/ds340-notebook-mpl")
os.environ.setdefault("XDG_CACHE_HOME", "/tmp/ds340-notebook-cache")
from extension.clustering.mispricing.structural import (  # noqa: E402 - repository path must be set before project imports
    Path,
    argparse,
    curve,
    equal_support_test,
    estimate,
    fingerprint,
    hashlib,
    importlib,
    json,
    pd,
    positions,
    structural_labels,
    threadpool_limits,
)

SOURCE_FILE = ROOT / "extension/clustering/mispricing/structural.py"

## 2. Configuration
Defaults match the existing CLI, except for isolated paths. Edit this argument list for a separate sensitivity run, using a new output directory. Do not select a model using the held-out results.

In [ ]:
NOTEBOOK_ARGUMENTS = [
    "--data",
    "data/kalshi_recent_year_csv",
    "--cache",
    "output/notebook_extension/mispricing",
    "--output",
    "output/notebook_extension/structural",
]

## Parse and validate settings

In [ ]:
parser = argparse.ArgumentParser(description=__doc__)
parser.add_argument("--data", type=Path, default=Path("data/kalshi_recent_year_csv"))
parser.add_argument("--cache", type=Path, default=Path("extension/clustering/mispricing/results"))
parser.add_argument("--output", type=Path, default=Path("extension/clustering/mispricing/results/structural"))
args = parser.parse_args(NOTEBOOK_ARGUMENTS)

# Keep experimental writes away from the established implementation and results.
prototype_root = (ROOT / "output/notebook_extension").resolve()
if prototype_root not in args.output.resolve().parents:
    raise ValueError("Choose an output subdirectory under output/notebook_extension/.")
args.output.mkdir(parents=True, exist_ok=True)
(args.output / "run_manifest.json").write_text(
    json.dumps({"completed": False, "interface": "experimental_notebook"}, indent=2)
)

## Verify the audited cache and assignments

In [ ]:
audit = json.loads((args.cache / "data_audit.json").read_text())
inputs = fingerprint([args.data / "master/markets.csv", args.data / "master/trades.csv"])
if inputs != audit["inputs"][:2]:
    raise ValueError("CSV inputs do not match audited statistics; rerun the base mispricing extraction first")
assignment_path = Path(audit["inputs"][2]["path"])
if fingerprint([assignment_path])[0] != audit["inputs"][2]:
    raise ValueError("Original assignments changed; refresh the audited statistics before population matching")

## Create the isolated run directory

In [ ]:
args.output.mkdir(parents=True, exist_ok=True)

## Record analysis provenance

In [ ]:
manifest = {
    "completed": False,
    "inputs": inputs,
    "cache": fingerprint([args.cache / "sufficient_statistics.csv.gz"]),
    "grouping": "rule-only domain/underlying/structure; no encoder or clustering model",
    "config": {k: str(v) for (k, v) in vars(args).items()},
}
manifest["completed"] = False
manifest["notebook_source_sha256"] = hashlib.sha256(
    (ROOT / "notebooks/extension_prototype/04_structural_families.ipynb").read_bytes()
).hexdigest()
manifest["interface"] = "experimental_notebook"
manifest_path = args.output / "run_manifest.json"
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")

## Rebuild rule-only family assignments

In [ ]:
print("Recomputing structural attributes from market CSV titles and YES subtitles...", flush=True)
markets = pd.read_csv(
    args.data / "master/markets.csv",
    dtype=str,
    keep_default_na=False,
    usecols=["ticker", "event_ticker", "market_type", "title", "yes_sub_title"],
)
labels = structural_labels(markets)
labels.to_csv(args.output / "structural_assignments.csv", index=False)
cells = pd.read_csv(args.cache / "sufficient_statistics.csv.gz", keep_default_na=False)
cells["previously_ml_assigned"] = cells.cluster.ne("")
cells = cells.drop(columns=["cluster", "family"]).merge(
    labels[["ticker", "structural_group"]], on="ticker", validate="many_to_one"
)
cells = cells.rename(columns={"structural_group": "cluster"})

## Initialize estimates

In [ ]:
summaries = []
tests = []
equality = []
pooled = []
coverage = []

## Compare structural populations and weightings

In [ ]:
with threadpool_limits(limits=2):
    for population in ["all_structural", "same_ml_population"]:
        subset = cells if population == "all_structural" else cells[cells.previously_ml_assigned]
        coverage.append(
            {
                "population": population,
                "contracts": subset.ticker.nunique(),
                "events": subset.event_id.nunique(),
                "executions": int(subset.n.sum()),
                "units": float(subset.units.sum()),
                "structural_groups": subset.cluster.nunique(),
            }
        )
        for weighting, column in [("execution", "n"), ("unit", "units")]:
            print(f"{population}, {weighting}: estimating wedges...", flush=True)
            pos = positions(subset, column)
            c = curve(pos, True)
            c["population"] = population
            c["weighting"] = weighting
            c.to_csv(args.output / f"curves_{population}_{weighting}.csv", index=False)
            p = curve(pos)
            p["population"] = population
            p["weighting"] = weighting
            pooled.append(p)
            bands = [((1, 49), 10)]
            if population == "all_structural":
                bands += [((1, 10), 6), ((11, 20), 6), ((21, 30), 6), ((31, 40), 6), ((41, 49), 6)]
            for band, mincost in bands:
                (result, test) = estimate(pos, band, mincost, 20, 10, weighting)
                result["population"] = population
                summaries.append(result)
                tests.append({"population": population, "weighting": weighting, "band": f"{band[0]}-{band[1]}", **test})
                if band == (1, 49):
                    for row in equal_support_test(pos, result):
                        equality.append({"population": population, "weighting": weighting, **row})

## Export estimates and coverage

In [ ]:
pd.concat(summaries, ignore_index=True).to_csv(args.output / "structural_summary.csv", index=False)
pd.DataFrame(tests).to_csv(args.output / "heterogeneity_tests.csv", index=False)
pd.DataFrame(equality).to_csv(args.output / "fixed_support_equality_tests.csv", index=False)
pd.DataFrame(coverage).to_csv(args.output / "population_coverage.csv", index=False)
pd.concat(pooled, ignore_index=True).to_csv(args.output / "pooled_curves.csv", index=False)

## Generate the isolated report and figures

In [ ]:
from extension.clustering.mispricing.structural_report import write_report

target = args.output / "STRUCTURAL_FINDINGS.md"
write_report(args.output, args.cache, target)

## Finalize the run

In [ ]:
manifest["source_hashes"] = {
    p.name: hashlib.sha256(p.read_bytes()).hexdigest() for p in Path(SOURCE_FILE).parent.glob("*.py")
}
manifest["features_sha256"] = hashlib.sha256(
    Path(SOURCE_FILE).parent.parent.joinpath("features.py").read_bytes()
).hexdigest()
manifest["versions"] = {
    name: importlib.metadata.version(name) for name in ["numpy", "pandas", "scipy", "duckdb", "matplotlib"]
}
manifest["completed"] = True
manifest_path.write_text(json.dumps(manifest, indent=2) + "\n")
print(f"Completed: {target}", flush=True)

## Inspect results and limitations
A completed manifest means execution finished; it does not validate the research assumptions. This is not a chronological backtest, a causal wording experiment, or an execution-aware trading simulation. Sparse/rejected groups must remain visible. Compare this run with the CLI using identical inputs, settings and software before adopting the notebook route.

In [ ]:
from IPython.display import Image, display

result_dir = args.output.resolve()
print("Results:", result_dir)
print("Completed:", json.loads((result_dir / "run_manifest.json").read_text()).get("completed", False))
for csv_name in ["model_comparison.csv", "coverage.csv", "population_coverage.csv", "heterogeneity_tests.csv"]:
    csv_path = result_dir / csv_name
    if csv_path.exists():
        print(csv_name)
        display(pd.read_csv(csv_path).head(20))
for image_path in sorted(result_dir.glob("*.png")) + sorted((result_dir / "figures").glob("*.png")):
    display(Image(filename=str(image_path)))
report_path = result_dir / "FINDINGS.md"
if not report_path.exists():
    report_path = result_dir / "STRUCTURAL_FINDINGS.md"
if report_path.exists():
    print("Report:", report_path)